# Numerical Simulation of Learning Dynamics in Multimodal RL

CSE 402 project notebook. Base paper: Sinha, Elango & Liu (2026), *Expected Return Causes
Outcome-Level Mode Collapse in Reinforcement Learning and How to Fix It with Inverse
Probability Scaling*, arXiv:2601.21669.

This notebook runs the experiment scripts from the repository and displays their figures.
Everything is pure NumPy/SciPy on the CPU — **no GPU is used or needed**; the state of the
system is a length-`K` logit vector.

## How to run this on Kaggle

1. Zip the repository directory and attach it as a Kaggle **Dataset**.
2. Import this file as a Kaggle **Notebook**.
3. Run the *Setup* cell below. It locates the code under `/kaggle/input/...`, copies it to a
   writable directory (the scripts write figures and CSVs next to themselves), and puts it on
   `sys.path`.
4. Run the experiment cell you want. Experiment 5 is the default; the others are there so the
   whole project can be reproduced from one notebook.

Accelerator: **None** is fine. Approximate CPU runtimes: Exp. 1 ≈ 2 min, Exp. 2 ≈ 3 min,
Exp. 3 ≈ 6 min, Exp. 4 ≈ 4 min, Exp. 5 ≈ 10 min.

## Setup

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path


def locate_project():
    """Find the repository root: the directory that contains src/ and experiments/.

    Works when the notebook sits inside the repo (local use) and when the repo has been
    attached as a Kaggle dataset under /kaggle/input/<name>/ (possibly one level deeper,
    if the zip contained a top-level folder).
    """
    candidates = [Path.cwd(), *Path.cwd().parents]
    for root in (Path('/kaggle/input'), Path('/kaggle/working')):
        if root.exists():
            candidates.append(root)
            candidates.extend(sorted(root.iterdir()))
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    candidates.extend(sorted(d for d in child.iterdir() if d.is_dir()))
    for path in candidates:
        try:
            if (path / 'src' / 'dynamics.py').is_file() and (path / 'experiments').is_dir():
                return path.resolve()
        except (OSError, PermissionError):
            continue
    raise FileNotFoundError(
        'Could not find the project. Attach the repository as a Kaggle dataset, or run this '
        'notebook from inside the repository directory.')


SOURCE = locate_project()
print('found project at:', SOURCE)

# Kaggle input is read-only and the scripts write results/ next to themselves, so work on a copy.
if str(SOURCE).startswith('/kaggle/input'):
    PROJECT = Path('/kaggle/working/alpha-ips-rl')
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(SOURCE, PROJECT)
    print('copied to writable location:', PROJECT)
else:
    PROJECT = SOURCE

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import matplotlib
import numpy
import scipy

print('python', sys.version.split()[0], '| numpy', numpy.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)
print('working directory:', Path.cwd())

In [ ]:
import json

from IPython.display import Image, Markdown, display


def run_experiment(script):
    """Run one experiment script as a subprocess, streaming its log."""
    path = PROJECT / 'experiments' / script
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f'running {script} ...\n' + '=' * 74)
    started = time.time()
    process = subprocess.Popen([sys.executable, '-u', str(path)], cwd=str(PROJECT),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    print('=' * 74)
    print(f'{script} finished with exit code {code} in {time.time() - started:.0f}s')
    if code != 0:
        raise RuntimeError(f'{script} failed (exit code {code})')


def show_figures(experiment):
    """Display every PNG produced by an experiment, in filename order."""
    folder = PROJECT / 'results' / experiment / 'figures'
    figures = sorted(folder.glob('fig*.png'))
    if not figures:
        print('no figures found in', folder)
    for figure in figures:
        display(Markdown(f'### `{figure.name}`'))
        display(Image(filename=str(figure)))


def show_summary(experiment, summary_file):
    path = PROJECT / 'results' / experiment / 'data' / summary_file
    if not path.is_file():
        print('no summary at', path)
        return None
    with open(path, encoding='utf-8') as handle:
        summary = json.load(handle)
    print(json.dumps(summary, indent=2)[:6000])
    return summary


def show_writeup(name):
    path = PROJECT / name
    if path.is_file():
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print('missing', path)

## Unit tests

Run these first: they check every identity, solver, optimiser and eigenvalue routine the
experiments rely on, and they take under a minute.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                         '-t', '.', '-v'],
                        cwd=str(PROJECT), capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-8000:])
print('exit code:', result.returncode)

## Experiment 5 — estimator bias, variance, and what they cost downstream

The project pitch's Section 4.4. How far is the empirical frequency $\hat p$ from the true $p$
with a group of $G$ samples, how do $\varepsilon$-clipping, Laplace smoothing, Richardson
extrapolation and a moving average trade bias against variance, and which of those trades the
learning dynamics actually notice. Derivations in `derivation_exp5.md`, write-up in
`RESULTS_exp5.md`.

Runtime ≈ 10 minutes on a Kaggle CPU.

In [ ]:
run_experiment('exp5_bias_variance.py')

In [ ]:
show_figures('exp5_bias_variance')

In [ ]:
summary5 = show_summary('exp5_bias_variance', 'exp5_summary.json')

In [ ]:
show_writeup('RESULTS_exp5.md')

## Experiments 1–4 (optional, for full reproduction)

Each cell is independent; run only the ones you need.

In [ ]:
run_experiment('exp1_collapse.py')
show_figures('exp1_collapse')

In [ ]:
run_experiment('exp2_ips_alpha_sweep.py')
show_figures('exp2_ips_alpha_sweep')

In [ ]:
run_experiment('exp3_rootfinding.py')
show_figures('exp3_rootfinding')

In [ ]:
run_experiment('exp4_hypergrid.py')
show_figures('exp4_hypergrid')

## Collect the outputs

Everything written by the scripts lives under `results/<experiment>/{figures,data}`. On Kaggle
that is inside `/kaggle/working`, so it is saved with the notebook version; this cell also zips
it for a single download.

In [ ]:
results = PROJECT / 'results'
for folder in sorted(p for p in results.iterdir() if p.is_dir()):
    figures = sorted((folder / 'figures').glob('*.png'))
    tables = sorted((folder / 'data').glob('*'))
    print(f'{folder.name:24s} {len(figures):2d} figures, '
          f'{len([t for t in tables if t.suffix in (".csv", ".json")]):2d} data files')

archive = shutil.make_archive(str(Path.cwd() / 'alpha_ips_rl_results'), 'zip', str(results))
print('\nwrote', archive, f'({os.path.getsize(archive) / 1e6:.1f} MB)')